In [3]:
import numpy as np
import matplotlib.pyplot as plt
import Tdoa
import Aoa
import Hybrid
import Comm as com
import Multipath as mp
import importlib
import util

In [5]:
plt.rcParams.update({
    "font.size": 13,
    "font.family": "serif",
    "mathtext.fontset": "cm",  # This gives the LaTeX look to math text
    "axes.edgecolor": "black", # Black bounding box
    "axes.linewidth": 0.8
})

In [6]:
num_rx = 4

In [ ]:
importlib.reload(Aoa)
importlib.reload(Tdoa)
importlib.reload(Hybrid)

# Read the taps computed by MUSIC from the CSV file
# And perform localization using TDOA, AOA, and Hybrid methods

tdoa_errors = []
aoa_errors = []
hybrid_errors = []
hybrid_ransac_errors = []
hybrid_ransac_fo_errors = []

for i in range(8):
    for j in range(5):
        filename = f"Small8/taps_estimates_{i}_{j}.csv"
        TX_position, RX_positions, RX_orientations, data = util.read_taps_from_csv(filename)

        for iter in range(1):

            taus_hat = data[f'{iter}']['delays']
            angles_hat = data[f'{iter}']['angles']

            first_toas = np.zeros(num_rx)
            for RX_idx in range(num_rx):
                first_toa = taus_hat[RX_idx][0] if len(taus_hat[RX_idx]) > 0 else -1
                first_toas[RX_idx] = first_toa

            first_aoas = np.zeros(num_rx)
            for RX_idx in range(num_rx):
                first_aoa = angles_hat[RX_idx][0] if len(angles_hat[RX_idx]) > 0 else 4
                first_aoas[RX_idx] = util.global_angle(first_aoa, RX_orientations[RX_idx])

            # TDOA localization using first path
            TdoaLocalizer = Tdoa.TdoaLocalization([rx[:2] for rx in RX_positions], first_toas)
            position_estimate_tdoa = TdoaLocalizer.localize()
            tdoa_errors.append(np.linalg.norm(position_estimate_tdoa - TX_position[:2]))

            # AOA localization using first path
            AoaLocalizer = Aoa.AoaLocalization(first_aoas, [rx[:2] for rx in RX_positions])
            position_estimate_aoa = AoaLocalizer.localize_least_squares()
            aoa_errors.append(np.linalg.norm(position_estimate_aoa - TX_position[:2]))

            # Hybrid localization
            HybridLocalizer = Hybrid.HybridLocalization([rx[:2] for rx in RX_positions], RX_orientations, taus_hat, angles_hat)
            position_estimate_hybrid = HybridLocalizer.localize()
            hybrid_errors.append(np.linalg.norm(position_estimate_hybrid - TX_position[:2]))

            # Hybrid localization with RANSAC
            position_estimate_hybrid_ransac = HybridLocalizer.localize_ransac(iter=100, threshold=0.5)
            hybrid_ransac_errors.append(np.linalg.norm(position_estimate_hybrid_ransac - TX_position[:2]))

            # Hybrid localization with RANSAC + first-order reflections
            try:
                position_estimate_hybrid_ransac_fo = HybridLocalizer.localize_iterative2(iter=100, threshold=0.5)
                hybrid_ransac_fo_errors.append(np.linalg.norm(position_estimate_hybrid_ransac_fo - TX_position[:2]))
            except Exception as e:
                print(f"Error occurred while localizing with RANSAC + FO: {e}")
                print(taus_hat)

KeyboardInterrupt: 

In [9]:
importlib.reload(Aoa)
importlib.reload(Tdoa)
importlib.reload(Hybrid)

# Read the taps computed by MUSIC from the CSV file
# And perform localization using TDOA, AOA, and Hybrid methods

tdoa_errors_4 = []
tdoa_errors_6 = []
tdoa_errors_8 = []

for i in range(8):
    for j in range(5):
        for num_rx in [4, 6, 8]:
            filename = f"Small{num_rx}/taps_estimates_{i}_{j}.csv"
            TX_position, RX_positions, RX_orientations, data = util.read_taps_from_csv(filename)

            for iter in range(10):
                
                print(num_rx, i, j, iter)
                taus_hat = data[f'{iter}']['delays']
                angles_hat = data[f'{iter}']['angles']

                first_toas = np.zeros(num_rx)
                for RX_idx in range(num_rx):
                    first_toa = taus_hat[RX_idx][0] if len(taus_hat[RX_idx]) > 0 else -1
                    first_toas[RX_idx] = first_toa

                first_aoas = np.zeros(num_rx)
                for RX_idx in range(num_rx):
                    first_aoa = angles_hat[RX_idx][0] if len(angles_hat[RX_idx]) > 0 else 4
                    first_aoas[RX_idx] = util.global_angle(first_aoa, RX_orientations[RX_idx])

                # TDOA localization using first path
                TdoaLocalizer = Tdoa.TdoaLocalization([rx[:2] for rx in RX_positions], first_toas)
                position_estimate_tdoa = TdoaLocalizer.localize()
                if num_rx == 4:
                    tdoa_errors_4.append(np.linalg.norm(position_estimate_tdoa - TX_position[:2]))
                elif num_rx == 6:
                    tdoa_errors_6.append(np.linalg.norm(position_estimate_tdoa - TX_position[:2]))
                elif num_rx == 8:
                    tdoa_errors_8.append(np.linalg.norm(position_estimate_tdoa - TX_position[:2]))


4 0 0 0
4 0 0 1
4 0 0 2
4 0 0 3
4 0 0 4
4 0 0 5
4 0 0 6
4 0 0 7
4 0 0 8
4 0 0 9
6 0 0 0
6 0 0 1
6 0 0 2
6 0 0 3
6 0 0 4
6 0 0 5
6 0 0 6
6 0 0 7
6 0 0 8
6 0 0 9
8 0 0 0
8 0 0 1
8 0 0 2
8 0 0 3
8 0 0 4


KeyError: '4'